# 🚀 FinBot - Setup Validation & Testing

Ce notebook valide que l'installation et la configuration du projet **Financial Market Analyzer** sont correctes. Il teste tous les composants essentiels avant de commencer le développement.

## 📋 Objectifs

1. ✅ Vérifier l'environnement Python
2. ✅ Valider les dépendances installées
3. ✅ Tester la configuration (variables d'environnement)
4. ✅ Vérifier les imports des modules
5. ✅ Tester la connectivité API (si clés disponibles)
6. ✅ Valider le système de cache
7. ✅ Confirmer la structure du projet

## 1️⃣ Vérification de l'Environnement Python

Vérifions la version de Python et les informations système.

In [ ]:
# Imports système
import sys
import platform
from datetime import datetime

# Afficher les informations système
print("=" * 60)
print("🖥️  INFORMATIONS SYSTÈME")
print("=" * 60)
print(f"Python Version    : {sys.version}")
print(f"Platform          : {platform.platform()}")
print(f"Architecture      : {platform.machine()}")
print(f"Processor         : {platform.processor()}")
print(f"Date/Time         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Vérifier la version de Python
required_version = (3, 11)
current_version = sys.version_info[:2]

if current_version >= required_version:
    print(f"✅ Python version OK (>= {required_version[0]}.{required_version[1]})")
else:
    print(f"❌ Python version insuffisante. Requis: >= {required_version[0]}.{required_version[1]}, Actuel: {current_version[0]}.{current_version[1]}")
    
print()

## 2️⃣ Vérification des Dépendances Installées

Vérifions que toutes les bibliothèques essentielles sont installées avec les bonnes versions.

In [ ]:
# Liste des packages essentiels à vérifier
essential_packages = {
    "pandas": "Data analysis",
    "numpy": "Numerical computing",
    "yfinance": "Yahoo Finance data",
    "requests": "HTTP requests",
    "python-dotenv": "Environment variables",
    "fastapi": "API framework",
    "pytest": "Testing framework"
}

print("=" * 60)
print("📦 VÉRIFICATION DES DÉPENDANCES")
print("=" * 60)

missing_packages = []
installed_packages = []

for package, description in essential_packages.items():
    try:
        # Gérer les cas spéciaux de noms de packages
        import_name = package.replace("-", "_")
        if package == "python-dotenv":
            import_name = "dotenv"
            
        module = __import__(import_name)
        version = getattr(module, "__version__", "version inconnue")
        print(f"✅ {package:20s} v{version:15s} - {description}")
        installed_packages.append(package)
    except ImportError:
        print(f"❌ {package:20s} {'MANQUANT':15s} - {description}")
        missing_packages.append(package)

print("=" * 60)
print(f"Installés : {len(installed_packages)}/{len(essential_packages)}")

if missing_packages:
    print(f"\n⚠️  Packages manquants : {', '.join(missing_packages)}")
    print("   Exécutez : pip install -r requirements.txt")
else:
    print("\n✅ Toutes les dépendances essentielles sont installées!")
    
print()

## 3️⃣ Configuration et Variables d'Environnement

Chargeons la configuration du projet et vérifions les variables d'environnement.

In [ ]:
import sys
import os
from pathlib import Path

# Ajouter le répertoire src au PYTHONPATH
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Added to path: {src_path}")
print()

# Charger la configuration
try:
    from financial_analyzer import config
    from dotenv import load_dotenv
    
    # Charger .env
    env_path = project_root / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print("✅ Fichier .env chargé")
    else:
        print("⚠️  Fichier .env non trouvé. Utilisation des valeurs par défaut.")
        print(f"   Créez .env depuis .env.example : cp {project_root}/.env.example {env_path}")
    
    print("\n" + "=" * 60)
    print("⚙️  CONFIGURATION DU PROJET")
    print("=" * 60)
    
    # Afficher les configurations
    print(f"\n📁 Chemins:")
    print(f"   BASE_DIR     : {config.BASE_DIR}")
    print(f"   DATA_DIR     : {config.DATA_DIR}")
    print(f"   CACHE_DIR    : {config.CACHE_DIR}")
    print(f"   LOGS_DIR     : {config.LOGS_DIR}")
    
    print(f"\n🔧 Configuration générale:")
    print(f"   ENVIRONMENT  : {config.ENVIRONMENT}")
    print(f"   LOG_LEVEL    : {config.LOG_LEVEL}")
    print(f"   CACHE_ENABLED: {config.CACHE_ENABLED}")
    
    print(f"\n🔑 API Keys (configurées):")
    for key, value in config.API_KEYS.items():
        status = "✅ Configurée" if value else "❌ Manquante"
        masked = value[:10] + "..." if value else "Non définie"
        print(f"   {key:25s}: {status:20s} ({masked})")
    
    print(f"\n🤖 Configuration ML:")
    for key, value in config.ML_CONFIG.items():
        print(f"   {key:20s}: {value}")
    
    print(f"\n📊 Configuration Trading:")
    for key, value in config.TRADING_CONFIG.items():
        print(f"   {key:20s}: {value}")
    
    print("\n✅ Configuration chargée avec succès!")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement de la configuration: {e}")
    print(f"   Vérifiez que le module config.py existe dans src/financial_analyzer/")
    
print()

## 4️⃣ Test des Imports des Modules

Vérifions que tous les modules du projet peuvent être importés sans erreur.

In [ ]:
print("=" * 60)
print("📚 TEST DES IMPORTS DES MODULES")
print("=" * 60)

modules_to_test = [
    ("financial_analyzer", "Package principal"),
    ("financial_analyzer.config", "Configuration"),
    ("financial_analyzer.utils", "Utilitaires"),
    ("financial_analyzer.utils.helpers", "Fonctions helpers"),
    ("financial_analyzer.data", "Module data"),
    ("financial_analyzer.sentiment", "Module sentiment"),
    ("financial_analyzer.analysis", "Module analysis"),
    ("financial_analyzer.recommendations", "Module recommendations"),
]

success_count = 0
failed_modules = []

for module_name, description in modules_to_test:
    try:
        __import__(module_name)
        print(f"✅ {module_name:45s} - {description}")
        success_count += 1
    except ImportError as e:
        print(f"❌ {module_name:45s} - {description}")
        print(f"   Erreur: {e}")
        failed_modules.append(module_name)

print("=" * 60)
print(f"Réussis : {success_count}/{len(modules_to_test)}")

if failed_modules:
    print(f"\n⚠️  Modules avec erreurs : {', '.join(failed_modules)}")
else:
    print("\n✅ Tous les modules peuvent être importés correctement!")
    
print()

## 5️⃣ Test des Utilitaires (helpers)

Testons les fonctions utilitaires du module `helpers`.

In [ ]:
from financial_analyzer.utils.helpers import (
    validate_ticker,
    validate_date,
    get_logger
)

print("=" * 60)
print("🧪 TEST DES FONCTIONS UTILITAIRES")
print("=" * 60)

# Test 1: Validation de tickers
print("\n1️⃣  Test validate_ticker():")
test_tickers = ["aapl", "MSFT", " GOOGL ", "TSLA"]
for ticker in test_tickers:
    try:
        validated = validate_ticker(ticker)
        print(f"   ✅ '{ticker}' -> '{validated}'")
    except ValueError as e:
        print(f"   ❌ '{ticker}' -> Erreur: {e}")

# Test 2: Validation de dates
print("\n2️⃣  Test validate_date():")
test_dates = ["2023-01-01", "2024-12-31", "2025-06-15"]
for date in test_dates:
    try:
        validated = validate_date(date)
        print(f"   ✅ '{date}' validée")
    except ValueError as e:
        print(f"   ❌ '{date}' -> Erreur: {e}")

# Test 3: Test de dates invalides
print("\n3️⃣  Test avec dates invalides (doit échouer):")
invalid_dates = ["2023/01/01", "01-01-2023", "invalid"]
for date in invalid_dates:
    try:
        validate_date(date)
        print(f"   ❌ '{date}' - Ne devrait pas être valide!")
    except ValueError:
        print(f"   ✅ '{date}' - Correctement rejetée")

# Test 4: Logger
print("\n4️⃣  Test get_logger():")
logger = get_logger("test_notebook")
logger.info("Test du logger - message INFO")
logger.warning("Test du logger - message WARNING")
print("   ✅ Logger créé et fonctionnel")

print("\n✅ Tous les tests d'utilitaires passés!")
print()

## 6️⃣ Test du Système de Cache

Testons le décorateur `@cache_result` pour vérifier que le système de cache fonctionne.

In [ ]:
import time
from financial_analyzer.utils.helpers import cache_result
from financial_analyzer.config import CACHE_DIR

print("=" * 60)
print("💾 TEST DU SYSTÈME DE CACHE")
print("=" * 60)

# Fonction de test avec cache
@cache_result("test_data_{param}", expiry_hours=1)
def expensive_function(param: str) -> dict:
    """Simule une fonction coûteuse."""
    print(f"   🔄 Calcul en cours pour '{param}'...")
    time.sleep(1)  # Simule un traitement long
    return {"param": param, "result": f"Données pour {param}", "timestamp": time.time()}

# Test 1: Premier appel (calcul)
print("\n1️⃣  Premier appel (devrait calculer):")
start = time.time()
result1 = expensive_function("AAPL")
duration1 = time.time() - start
print(f"   Résultat: {result1['result']}")
print(f"   Durée: {duration1:.2f}s")

# Test 2: Deuxième appel (depuis cache)
print("\n2️⃣  Deuxième appel (devrait utiliser le cache):")
start = time.time()
result2 = expensive_function("AAPL")
duration2 = time.time() - start
print(f"   Résultat: {result2['result']}")
print(f"   Durée: {duration2:.2f}s")

if duration2 < duration1 / 2:
    print("   ✅ Cache fonctionnel (beaucoup plus rapide!)")
else:
    print("   ⚠️  Le cache ne semble pas fonctionner")

# Test 3: Vérifier les fichiers de cache
print("\n3️⃣  Fichiers de cache créés:")
cache_files = list(CACHE_DIR.glob("*.pkl"))
if cache_files:
    print(f"   ✅ {len(cache_files)} fichier(s) de cache trouvé(s)")
    for cache_file in cache_files[:5]:  # Afficher les 5 premiers
        print(f"      - {cache_file.name}")
else:
    print("   ⚠️  Aucun fichier de cache trouvé")

print("\n✅ Test du système de cache terminé!")
print()

## 7️⃣ Validation de la Structure du Projet

Vérifions que tous les dossiers et fichiers essentiels sont en place.

In [ ]:
print("=" * 60)
print("📁 VALIDATION DE LA STRUCTURE DU PROJET")
print("=" * 60)

# Dossiers essentiels
essential_dirs = [
    "src",
    "src/financial_analyzer",
    "src/financial_analyzer/data",
    "src/financial_analyzer/sentiment",
    "src/financial_analyzer/analysis",
    "src/financial_analyzer/recommendations",
    "src/financial_analyzer/utils",
    "api",
    "api/routes",
    "notebooks",
    "tests",
    "docs",
    "data",
    "data/raw",
    "data/processed",
    "data/cache",
    "logs",
    "scripts",
    ".github"
]

# Fichiers essentiels
essential_files = [
    "README.md",
    "requirements.txt",
    "setup.py",
    ".gitignore",
    ".env.example",
    "Makefile",
    "pytest.ini",
    "LICENSE",
    "CONTRIBUTING.md",
    ".github/copilot-instructions.md",
    "src/financial_analyzer/config.py",
    "src/financial_analyzer/utils/helpers.py"
]

print("\n📂 Vérification des dossiers:")
missing_dirs = []
for dir_path in essential_dirs:
    full_path = project_root / dir_path
    if full_path.exists() and full_path.is_dir():
        print(f"   ✅ {dir_path}")
    else:
        print(f"   ❌ {dir_path}")
        missing_dirs.append(dir_path)

print("\n📄 Vérification des fichiers:")
missing_files = []
for file_path in essential_files:
    full_path = project_root / file_path
    if full_path.exists() and full_path.is_file():
        size = full_path.stat().st_size
        print(f"   ✅ {file_path:50s} ({size:,} bytes)")
    else:
        print(f"   ❌ {file_path}")
        missing_files.append(file_path)

print("\n" + "=" * 60)
print(f"Dossiers : {len(essential_dirs) - len(missing_dirs)}/{len(essential_dirs)}")
print(f"Fichiers : {len(essential_files) - len(missing_files)}/{len(essential_files)}")

if missing_dirs or missing_files:
    print("\n⚠️  Éléments manquants:")
    if missing_dirs:
        print(f"   Dossiers: {', '.join(missing_dirs)}")
    if missing_files:
        print(f"   Fichiers: {', '.join(missing_files)}")
else:
    print("\n✅ Structure du projet complète et valide!")
    
print()

## 🎉 Résumé Final

Récapitulons l'état du projet.

In [ ]:
print("=" * 60)
print("🎉 RÉSUMÉ DE LA VALIDATION DU SETUP")
print("=" * 60)

print("\n✅ **VALIDÉ:**")
print("   • Environnement Python 3.11+")
print("   • Configuration centralisée (config.py)")
print("   • Module utils.helpers fonctionnel")
print("   • Système de cache opérationnel")
print("   • Structure de projet complète")
print("   • Imports de tous les modules")

print("\n📋 **PROCHAINES ÉTAPES:**")
print("   1️⃣  Éditer .env avec vos API keys")
print("   2️⃣  Exécuter les tests: make test ou pytest tests/")
print("   3️⃣  Commencer la Semaine 1: Module DATA")
print("       → Créer src/financial_analyzer/data/market_data.py")
print("       → Implémenter MarketDataFetcher class")

print("\n💡 **COMMANDES UTILES:**")
print("   make install      # Installer les dépendances")
print("   make test         # Lancer les tests")
print("   make format       # Formater le code (black)")
print("   make lint         # Vérifier le code (flake8, mypy)")
print("   make notebook     # Lancer Jupyter")

print("\n🚀 **READY TO CODE!**")
print("=" * 60)